# Neural Membrane Dynamics: From Biophysics to Dynamical Systems

*A first-principles exploration of the Hodgkin-Huxley model*

## Why Model Neurons Mathematically?

The nervous system is arguably the most complex structure in the known universe. A single human brain contains roughly $10^{11}$ neurons, each forming thousands of synaptic connections. Yet the fundamental electrical event underlying all neural computation — the **action potential** — can be described by a system of just four ordinary differential equations. This remarkable compression of biological complexity into mathematical form is one of the great triumphs of 20th-century biophysics.

Mathematical models of neurons serve multiple purposes. First, they provide a **quantitative framework** for understanding how ionic currents flowing through protein channels in the cell membrane give rise to the rapid, all-or-nothing voltage spikes that carry information along nerve fibers. Without equations, we are limited to qualitative descriptions; with them, we can predict precisely how a neuron will respond to any given stimulus. Second, neural models reveal **universal dynamical principles** — bifurcations, limit cycles, excitability thresholds — that transcend the specific biological details and connect neuroscience to the broader theory of nonlinear dynamical systems.

Third, and perhaps most importantly, mathematical neuron models are the **building blocks of computational neuroscience**. From small circuits of a few neurons to large-scale brain simulations involving millions of units, every computational model of neural activity rests on some mathematical description of individual cells. Understanding the Hodgkin-Huxley model is therefore not merely an exercise in biophysics — it is the foundation upon which modern theoretical neuroscience is built.

### Historical Context

In the late 1940s and early 1950s, Alan Hodgkin and Andrew Huxley carried out a series of extraordinary experiments on the **giant axon of the Atlantic squid** (*Loligo*). The squid giant axon, with a diameter of up to 1 mm, was large enough to allow insertion of electrodes directly into the interior of the nerve fiber — a feat impossible with the much thinner axons of vertebrate neurons at the time.

Using the **voltage clamp technique**, Hodgkin and Huxley were able to hold the membrane potential at a fixed value and measure the ionic currents flowing through the membrane at that voltage. By systematically varying the clamp voltage and using ion substitution experiments (replacing external sodium with choline), they separated the total membrane current into its sodium, potassium, and leak components. They then fitted empirical equations to describe how the conductance of each ion channel depended on voltage and time.

The result, published in a landmark series of five papers in 1952, was a mathematical model that could **quantitatively reproduce** the shape, amplitude, duration, and propagation velocity of the action potential — all from first principles, without any free parameters adjusted after the fact. For this work, Hodgkin and Huxley were awarded the **Nobel Prize in Physiology or Medicine in 1963**.

### Roadmap

In this notebook, we will:

1. **Derive the electrical circuit model** of the neuronal membrane from physical principles
2. **Develop the Hodgkin-Huxley equations** step by step, motivating every term
3. **Simulate action potentials** and explore how they depend on stimulus parameters
4. **Analyze the ionic currents** underlying each phase of the action potential
5. **Investigate the dynamical systems structure** — fixed points, stability, and bifurcations
6. **Construct phase portraits** and nullcline diagrams
7. **Compute the frequency-current (f-I) relationship** and explore neural coding
8. **Compare with simplified models** such as the leaky integrate-and-fire neuron

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Allow imports from the src/ package
sys.path.insert(0, '../src')

from neural_dynamics import (
    alpha_n, beta_n, alpha_m, beta_m, alpha_h, beta_h,
    n_inf, m_inf, h_inf, tau_n, tau_m, tau_h,
    HH_DEFAULT_PARAMS,
    hh_rhs_parameterized, hh_initial_state, make_hh_rhs,
    solve_ode,
)

print("All imports successful.")

In [ ]:
# Publication-quality matplotlib configuration
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'lines.linewidth': 2,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Consistent color scheme for the entire notebook
COLORS = {
    'Na': '#E74C3C',     # red — sodium
    'K': '#3498DB',      # blue — potassium
    'leak': '#95A5A6',   # gray — leak
    'Vm': '#2C3E50',     # dark — membrane potential
    'accent': '#8E44AD', # purple — accent for special curves
    'green': '#27AE60',  # green — auxiliary
}

print("Matplotlib configured for publication-quality figures.")

---

# 2. The Neuron as an Electrical Circuit

To understand how a neuron generates electrical signals, we must first understand the **physical properties of the cell membrane** and how they give rise to electrical behavior. In this section, we build up the equivalent circuit model of a small patch of neuronal membrane, deriving each component from first principles.

## 2.1 The Cell Membrane as a Capacitor

The neuronal cell membrane is a **lipid bilayer** — a thin sheet (approximately $d \approx 7\text{–}8 \; \text{nm}$ thick) composed of phospholipid molecules whose hydrophobic tails face inward, forming a nearly impermeable barrier to ions. On either side of this insulating layer lies an electrically conducting aqueous solution: the **intracellular fluid** (cytoplasm) and the **extracellular fluid**.

This arrangement — two conductors separated by an insulating dielectric — is precisely the definition of a **parallel-plate capacitor**. From elementary electrostatics, the capacitance of a parallel-plate capacitor with plate area $A$, plate separation $d$, and dielectric constant $\varepsilon$ is:

$$C = \varepsilon \frac{A}{d} = \varepsilon_0 \varepsilon_r \frac{A}{d}$$

where $\varepsilon_0 \approx 8.85 \times 10^{-12} \; \text{F/m}$ is the permittivity of free space and $\varepsilon_r \approx 2$ is the relative permittivity of the lipid core. Substituting typical values:

$$c_m = \frac{C}{A} = \frac{\varepsilon_0 \varepsilon_r}{d} \approx \frac{(8.85 \times 10^{-12})(2)}{7.5 \times 10^{-9}} \approx 2.4 \times 10^{-3} \; \text{F/m}^2 \approx 0.24 \; \mu\text{F/cm}^2$$

Experimentally, the specific membrane capacitance is measured to be:

$$\boxed{C_m \approx 1 \; \mu\text{F/cm}^2}$$

The discrepancy (roughly a factor of 4) arises because the effective dielectric constant of the full membrane — including polar head groups, bound water molecules, and membrane proteins — is higher than that of the pure lipid core. The value $C_m \approx 1 \; \mu\text{F/cm}^2$ is remarkably universal across cell types and organisms.

The charge stored on the membrane capacitor is $Q = C_m V_m$, where $V_m = V_\text{in} - V_\text{out}$ is the **membrane potential**. The capacitive current (current flowing to charge or discharge the capacitor) is therefore:

$$\boxed{I_C = C_m \frac{dV_m}{dt}}$$

This tells us that the membrane potential can change only when current flows onto or off of the membrane capacitor. This is the starting point for all electrical models of neurons.

## 2.2 Ion Channels as Conductances

While the lipid bilayer is an excellent insulator, the cell membrane is studded with **ion channel proteins** — large transmembrane proteins that form aqueous pores through which specific ions can flow. The key channel types for the action potential are:

- **Voltage-gated Na$^+$ channels**: selectively permeable to sodium ions, open rapidly upon depolarization
- **Voltage-gated K$^+$ channels**: selectively permeable to potassium ions, open more slowly upon depolarization
- **Leak channels**: always open, primarily permeable to K$^+$ and Cl$^-$, responsible for the resting potential

Each population of ion channels can be characterized by a **conductance** $g_\text{ion}$ (the inverse of resistance), measured in units of mS/cm$^2$ (millisiemens per square centimeter of membrane). The conductance represents how easily ions can flow through the channels: high conductance means many channels are open, low conductance means few are open.

The current through a given ion channel population obeys an **Ohmic** (linear) relationship:

$$\boxed{I_\text{ion} = g_\text{ion} \cdot (V_m - E_\text{ion})}$$

where $E_\text{ion}$ is the **reversal potential** (or equilibrium potential) of that ion species. The driving force $(V_m - E_\text{ion})$ is the difference between the actual membrane potential and the potential at which the net current through those channels would be zero. When $V_m = E_\text{ion}$, the electrical and chemical driving forces on the ion are exactly balanced, and no net current flows — hence the name "reversal potential."

The critical insight that distinguishes the Hodgkin-Huxley model from simpler models is that the conductances $g_\text{Na}$ and $g_\text{K}$ are **not constant** — they are functions of both voltage and time. Understanding how to describe this voltage- and time-dependence is the central challenge of the model, and we will address it in Section 3.

## 2.3 The Nernst Equation (Full Derivation)

Where does the reversal potential $E_\text{ion}$ come from? It arises from the fact that ion concentrations differ dramatically between the inside and outside of the cell. For example, K$^+$ is roughly 20 times more concentrated inside the cell than outside, while Na$^+$ is roughly 10 times more concentrated outside. These concentration gradients are maintained by metabolic pumps (primarily the Na$^+$/K$^+$-ATPase).

When a channel selective for a particular ion opens, two forces act on that ion:

1. **The chemical force** (diffusion): ions tend to flow down their concentration gradient, from high to low concentration.
2. **The electrical force** (drift): ions are charged particles and are attracted or repelled by the electric field across the membrane.

At equilibrium, these two forces balance exactly. Let us derive the voltage at which this occurs.

### Derivation from Thermodynamic First Principles

Consider a single ion species with valence $z$ (e.g., $z = +1$ for Na$^+$ and K$^+$, $z = -1$ for Cl$^-$). The **electrochemical potential** of this ion on each side of the membrane has two contributions:

$$\mu = \mu_\text{chemical} + \mu_\text{electrical}$$

The chemical potential of an ideal solute at concentration $[\text{ion}]$ is:

$$\mu_\text{chemical} = \mu^0 + RT \ln[\text{ion}]$$

where $R = 8.314 \; \text{J/(mol} \cdot \text{K)}$ is the gas constant, $T$ is absolute temperature, and $\mu^0$ is a reference chemical potential.

The electrical potential energy of one mole of ions with charge $z$ at voltage $V$ is:

$$\mu_\text{electrical} = zFV$$

where $F = 96{,}485 \; \text{C/mol}$ is Faraday's constant. Therefore the total electrochemical potential is:

$$\mu = \mu^0 + RT \ln[\text{ion}] + zFV$$

At **thermodynamic equilibrium**, the electrochemical potential must be equal on both sides of the membrane:

$$\mu_\text{out} = \mu_\text{in}$$

$$\mu^0 + RT \ln[\text{ion}]_\text{out} + zF V_\text{out} = \mu^0 + RT \ln[\text{ion}]_\text{in} + zF V_\text{in}$$

The reference potentials $\mu^0$ cancel. Rearranging to solve for the transmembrane potential $E_\text{ion} = V_\text{in} - V_\text{out}$:

$$zF(V_\text{in} - V_\text{out}) = RT \ln[\text{ion}]_\text{out} - RT \ln[\text{ion}]_\text{in}$$

$$E_\text{ion} = V_\text{in} - V_\text{out} = \frac{RT}{zF} \ln \frac{[\text{ion}]_\text{out}}{[\text{ion}]_\text{in}}$$

This is the **Nernst equation**:

$$\boxed{E_\text{ion} = \frac{RT}{zF} \ln \frac{[\text{ion}]_\text{out}}{[\text{ion}]_\text{in}}}$$

At body temperature ($T = 310 \; \text{K}$) or squid axon temperature ($T \approx 279 \; \text{K}$ for $6.3°\text{C}$), the prefactor $RT/F \approx 24\text{–}27 \; \text{mV}$ for monovalent ions. Using the ionic concentrations of the squid giant axon:

| Ion | $[\text{ion}]_\text{in}$ (mM) | $[\text{ion}]_\text{out}$ (mM) | $z$ | $E_\text{ion}$ (mV) |
|-----|------|------|---|------|
| K$^+$ | 400 | 20 | +1 | $\approx -77$ |
| Na$^+$ | 50 | 440 | +1 | $\approx +50$ |
| Cl$^-$ | 40 | 560 | $-1$ | $\approx -66$ |

The large difference between $E_\text{Na} \approx +50 \; \text{mV}$ and $E_\text{K} \approx -77 \; \text{mV}$ is what makes the action potential possible: the membrane can swing between these two "battery" voltages by selectively opening Na$^+$ or K$^+$ channels.

In [ ]:
# Nernst equation: E_ion as a function of concentration ratio for different valences

R = 8.314    # J/(mol*K)
T = 279.45   # K (6.3°C, Hodgkin-Huxley experimental temperature)
F = 96485.0  # C/mol

# Concentration ratio [out]/[in]
ratio = np.logspace(-2, 2, 500)  # from 0.01 to 100

fig, ax = plt.subplots(figsize=(10, 6))

for z, label, ls in [(1, '$z = +1$ (Na$^+$, K$^+$)', '-'),
                      (2, '$z = +2$ (Ca$^{2+}$)', '--'),
                      (-1, '$z = -1$ (Cl$^-$)', '-.')]:
    E = (R * T / (z * F)) * np.log(ratio) * 1000  # convert V to mV
    ax.plot(ratio, E, ls, label=label, linewidth=2.5)

# Mark the actual squid axon values
E_K = (R * T / F) * np.log(20 / 400) * 1000
E_Na = (R * T / F) * np.log(440 / 50) * 1000
ax.plot(20 / 400, E_K, 'o', color=COLORS['K'], markersize=10, zorder=5,
        label=f'K$^+$ squid axon ({E_K:.0f} mV)')
ax.plot(440 / 50, E_Na, 's', color=COLORS['Na'], markersize=10, zorder=5,
        label=f'Na$^+$ squid axon ({E_Na:.0f} mV)')

ax.set_xscale('log')
ax.set_xlabel('Concentration ratio $[\\mathrm{ion}]_{\\mathrm{out}} / [\\mathrm{ion}]_{\\mathrm{in}}$')
ax.set_ylabel('Nernst potential $E_{\\mathrm{ion}}$ (mV)')
ax.set_title('Nernst Equation: Equilibrium Potential vs. Concentration Ratio')
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.axvline(1, color='gray', linewidth=0.8, linestyle=':')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(-200, 200)
ax.set_xlim(0.01, 100)

plt.tight_layout()
plt.show()

## 2.4 The Goldman-Hodgkin-Katz Equation

The Nernst equation gives the equilibrium potential for a single ion species. But at rest, the membrane is permeable to **multiple ion species simultaneously** — primarily K$^+$, Na$^+$, and Cl$^-$. The resting membrane potential is therefore not equal to any single Nernst potential, but rather a weighted average that depends on the **relative permeabilities** of the membrane to each ion.

The **Goldman-Hodgkin-Katz (GHK) voltage equation** generalizes the Nernst equation to multiple ions. It is derived from the Nernst-Planck equation for electrodiffusion under the assumption of a constant electric field across the membrane (the "constant field" approximation). For Na$^+$, K$^+$, and Cl$^-$:

$$\boxed{V_m = \frac{RT}{F} \ln \frac{P_\text{K}[\text{K}^+]_\text{out} + P_\text{Na}[\text{Na}^+]_\text{out} + P_\text{Cl}[\text{Cl}^-]_\text{in}}{P_\text{K}[\text{K}^+]_\text{in} + P_\text{Na}[\text{Na}^+]_\text{in} + P_\text{Cl}[\text{Cl}^-]_\text{out}}}$$

where $P_\text{K}$, $P_\text{Na}$, $P_\text{Cl}$ are the membrane permeabilities to each ion. Note that chloride concentrations are "flipped" (inside in numerator, outside in denominator) because Cl$^-$ has negative valence.

At rest, the squid axon membrane is much more permeable to K$^+$ than to Na$^+$, with typical permeability ratios $P_\text{K} : P_\text{Na} : P_\text{Cl} \approx 1 : 0.04 : 0.45$. Substituting the squid axon concentrations:

$$V_\text{rest} \approx -65 \; \text{mV}$$

This value is close to $E_\text{K}$ (since K$^+$ permeability dominates at rest) but is pulled slightly positive by the small Na$^+$ permeability. During an action potential, when Na$^+$ channels open and $P_\text{Na}$ increases dramatically, the GHK equation predicts that $V_m$ will swing toward $E_\text{Na} \approx +50 \; \text{mV}$.

## 2.5 The Equivalent Circuit

We can now assemble the complete electrical equivalent circuit for a patch of neuronal membrane. The membrane consists of:

1. A **capacitor** $C_m$ (the lipid bilayer)
2. Three parallel **conductance branches**, one for each ion species, each consisting of a variable resistor (the channels) in series with a battery (the Nernst potential)
3. An optional **external current source** $I_\text{ext}$ (representing synaptic input or an experimentalist's electrode)

By **Kirchhoff's current law**, the total current flowing into the membrane patch must be zero. All current that enters as $I_\text{ext}$ must either charge the capacitor or flow out through the ion channels:

$$I_\text{ext} = I_C + I_\text{Na} + I_\text{K} + I_L$$

Substituting the expressions for each current:

$$I_\text{ext} = C_m \frac{dV_m}{dt} + g_\text{Na}(V_m - E_\text{Na}) + g_\text{K}(V_m - E_\text{K}) + g_L(V_m - E_L)$$

Rearranging to express the dynamics of the membrane potential:

$$\boxed{C_m \frac{dV_m}{dt} = -g_\text{Na}(V_m - E_\text{Na}) - g_\text{K}(V_m - E_\text{K}) - g_L(V_m - E_L) + I_\text{ext}}$$

**This is THE fundamental equation of the Hodgkin-Huxley model.** Everything that follows — the gating variables, the rate functions, the full system of ODEs — is devoted to specifying how the conductances $g_\text{Na}$ and $g_\text{K}$ depend on voltage and time. The leak conductance $g_L$ is constant (representing channels that are always open).

The physical content of this equation is transparent:
- The left side is the rate of change of charge on the membrane capacitor.
- The right side is the net current: external current minus the sum of all ionic currents.
- Each ionic current is proportional to its conductance times its driving force.
- The membrane potential $V_m$ evolves toward whichever reversal potential has the largest conductance at that moment.

With constant conductances, this equation describes a simple leaky integrator — the membrane potential exponentially relaxes toward a weighted average of the reversal potentials. The action potential emerges only when we allow $g_\text{Na}$ and $g_\text{K}$ to depend on voltage, creating a powerful positive feedback loop. This is the subject of the next section.

---

# 3. Deriving the Hodgkin-Huxley Equations

In Section 2 we established that the membrane potential obeys $C_m \, dV/dt = -\sum I_\text{ion} + I_\text{ext}$, with ionic currents of the form $I_\text{ion} = g_\text{ion}(V_m - E_\text{ion})$. The remaining challenge is to describe how the conductances $g_\text{Na}$ and $g_\text{K}$ depend on voltage and time. This is where Hodgkin and Huxley's experimental genius and mathematical creativity come together.

## 3.1 Voltage-Dependent Conductances

Hodgkin and Huxley's voltage clamp experiments revealed a crucial fact: **the membrane conductances are not constant.** When the membrane is depolarized (voltage stepped to a more positive value):

- The **Na$^+$ conductance** rises rapidly to a peak and then decays back toward zero — even though the voltage is held constant. This transient behavior reflects two distinct processes: fast **activation** (opening) followed by slower **inactivation** (a conformational change that blocks the open channel).
- The **K$^+$ conductance** rises more slowly to a sustained plateau and remains elevated as long as the depolarization is maintained. K$^+$ channels activate but do not inactivate on the timescale of the action potential.

These are the experimental signatures that the model must capture. The conductances are functions of both voltage (because the rate of opening/closing depends on $V_m$) and time (because the channels take time to respond to voltage changes).

## 3.2 Gating Variables

Hodgkin and Huxley's key insight was to decompose each conductance into the product of a **maximum conductance** $\bar{g}$ (when all channels are open) and one or more **gating variables** that represent the fraction of channels in the open state. Each gating variable takes values between 0 (all gates closed) and 1 (all gates open).

### The K$^+$ Channel: Four Identical Gates ($n^4$)

The K$^+$ conductance rises sigmoidally upon depolarization — not exponentially. Hodgkin and Huxley found that the time course of K$^+$ activation could be well described by the **fourth power** of a single gating variable $n$:

$$\boxed{g_\text{K}(t) = \bar{g}_\text{K} \cdot n(t)^4}$$

Why $n^4$? The physical picture is that the K$^+$ channel has **four identical, independent subunits** (gates), each of which must be in the "open" configuration for the channel to conduct. If each gate has probability $n$ of being open (independently), then the probability that all four are open simultaneously is $n^4$. This explains the sigmoidal rise: at early times, $n$ is increasing from a small value, and $n^4$ increases even more steeply. We now know from molecular biology that voltage-gated K$^+$ channels are indeed tetramers — four identical $\alpha$-subunits arranged around a central pore — a remarkable confirmation of HH's purely mathematical deduction.

### The Na$^+$ Channel: Three Activation Gates and One Inactivation Gate ($m^3 h$)

The Na$^+$ conductance has a more complex time course: it rises rapidly (activation) and then decays (inactivation), even at constant voltage. Hodgkin and Huxley modeled this by introducing **two types of gates**:

- **$m$** — an activation gate. Three identical $m$-gates must be open for the channel to conduct (similar logic to $n$ for K$^+$).
- **$h$** — an inactivation gate. This single gate must also be open. Unlike $m$, the $h$-gate is open at rest and *closes* upon depolarization.

The Na$^+$ conductance is therefore:

$$\boxed{g_\text{Na}(t) = \bar{g}_\text{Na} \cdot m(t)^3 \cdot h(t)}$$

Why $m^3 h$? The $m^3$ factor produces a rapid, sigmoidal rise in conductance (since $m$ activates quickly upon depolarization). The $h$ factor produces a slower decay (since $h$ decreases upon depolarization, eventually shutting off the channel). The product $m^3 h$ thus naturally produces a transient conductance that rises fast and falls slowly — exactly as observed experimentally.

Molecularly, we now know that Na$^+$ channels are single large proteins with four homologous domains (I–IV). Three of these contribute to activation (analogous to the three $m$-gates), while a cytoplasmic loop between domains III and IV acts as the "inactivation ball" that plugs the open channel (analogous to the $h$-gate).

## 3.3 First-Order Kinetics

Each gating variable $x \in \{n, m, h\}$ represents the probability that a single gate is in the open state. A gate can transition between two states:

$$\text{Closed} \underset{\beta_x(V)}{\overset{\alpha_x(V)}{\rightleftharpoons}} \text{Open}$$

where $\alpha_x(V)$ is the voltage-dependent **opening rate** (transition rate from closed to open, in ms$^{-1}$) and $\beta_x(V)$ is the **closing rate** (open to closed). If $x$ is the fraction of gates in the open state (and $1-x$ is the fraction closed), the rate of change is:

$$\boxed{\frac{dx}{dt} = \alpha_x(V)(1 - x) - \beta_x(V) \, x}$$

This is a **first-order linear ODE** in $x$ (for fixed $V$). The first term describes gates opening (proportional to the fraction that are closed, $1-x$), and the second describes gates closing (proportional to the fraction that are open, $x$).

### Equivalent Formulation: Steady-State and Time Constant

The rate equation can be rewritten in a more physically transparent form. Defining:

$$x_\infty(V) = \frac{\alpha_x(V)}{\alpha_x(V) + \beta_x(V)} \qquad \text{and} \qquad \tau_x(V) = \frac{1}{\alpha_x(V) + \beta_x(V)}$$

we can verify by substitution that the ODE becomes:

$$\boxed{\frac{dx}{dt} = \frac{x_\infty(V) - x}{\tau_x(V)}}$$

This form makes the physical meaning completely clear:

- $x_\infty(V)$ is the **steady-state value** of $x$ at voltage $V$. If the voltage is held constant, $x$ will exponentially relax toward $x_\infty(V)$.
- $\tau_x(V)$ is the **time constant** of this relaxation. A small $\tau_x$ means the gate responds quickly to voltage changes; a large $\tau_x$ means it responds slowly.

The solution at constant voltage is:

$$x(t) = x_\infty - (x_\infty - x_0) \, e^{-t/\tau_x}$$

where $x_0 = x(0)$ is the initial value. This exponential relaxation is the signature behavior of HH gating variables.

## 3.4 The Rate Functions

Hodgkin and Huxley determined the rate functions $\alpha_x(V)$ and $\beta_x(V)$ by fitting empirical expressions to their voltage clamp data. The original paper used a shifted voltage variable $v = V - V_\text{rest}$, but we present the equations in the **modern convention** where $V$ is the absolute membrane potential (rest $\approx -65 \; \text{mV}$):

### K$^+$ activation ($n$)

$$\alpha_n(V) = \frac{0.01 \, (V + 55)}{1 - \exp\!\big(-(V+55)/10\big)} \qquad \beta_n(V) = 0.125 \, \exp\!\big(-(V+65)/80\big)$$

### Na$^+$ activation ($m$)

$$\alpha_m(V) = \frac{0.1 \, (V + 40)}{1 - \exp\!\big(-(V+40)/10\big)} \qquad \beta_m(V) = 4 \, \exp\!\big(-(V+65)/18\big)$$

### Na$^+$ inactivation ($h$)

$$\alpha_h(V) = 0.07 \, \exp\!\big(-(V+65)/20\big) \qquad \beta_h(V) = \frac{1}{1 + \exp\!\big(-(V+35)/10\big)}$$

### Singularity Handling

Note that $\alpha_n$ and $\alpha_m$ have the form $f(V) = a(V - V_0) / (1 - e^{-(V-V_0)/k})$, which is indeterminate ($0/0$) at $V = V_0$ (i.e., at $V = -55 \; \text{mV}$ for $\alpha_n$ and $V = -40 \; \text{mV}$ for $\alpha_m$). Applying **L'Hopital's rule**:

$$\lim_{V \to V_0} \frac{a(V - V_0)}{1 - e^{-(V-V_0)/k}} = \lim_{V \to V_0} \frac{a}{e^{-(V-V_0)/k} / k} = a \cdot k$$

So $\alpha_n(-55) = 0.01 \times 10 = 0.1$ and $\alpha_m(-40) = 0.1 \times 10 = 1.0$. Our numerical implementation handles this by checking whether $|V - V_0| < \epsilon$ and returning the limiting value directly.

In [ ]:
# Plot all 6 rate functions vs voltage (2x3 subplot grid)

V = np.linspace(-100, 50, 1000)

# Evaluate rate functions (vectorize since they are njit scalar functions)
an = np.array([alpha_n(v) for v in V])
bn = np.array([beta_n(v) for v in V])
am = np.array([alpha_m(v) for v in V])
bm = np.array([beta_m(v) for v in V])
ah = np.array([alpha_h(v) for v in V])
bh = np.array([beta_h(v) for v in V])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Row 1: alpha functions
axes[0, 0].plot(V, an, color=COLORS['K'], linewidth=2.5)
axes[0, 0].set_title(r'$\alpha_n(V)$ — K$^+$ activation opening')
axes[0, 0].set_ylabel('Rate (ms$^{-1}$)')

axes[0, 1].plot(V, am, color=COLORS['Na'], linewidth=2.5)
axes[0, 1].set_title(r'$\alpha_m(V)$ — Na$^+$ activation opening')

axes[0, 2].plot(V, ah, color=COLORS['accent'], linewidth=2.5)
axes[0, 2].set_title(r'$\alpha_h(V)$ — Na$^+$ inactivation recovery')

# Row 2: beta functions
axes[1, 0].plot(V, bn, color=COLORS['K'], linewidth=2.5, linestyle='--')
axes[1, 0].set_title(r'$\beta_n(V)$ — K$^+$ activation closing')
axes[1, 0].set_ylabel('Rate (ms$^{-1}$)')
axes[1, 0].set_xlabel('Membrane potential $V$ (mV)')

axes[1, 1].plot(V, bm, color=COLORS['Na'], linewidth=2.5, linestyle='--')
axes[1, 1].set_title(r'$\beta_m(V)$ — Na$^+$ activation closing')
axes[1, 1].set_xlabel('Membrane potential $V$ (mV)')

axes[1, 2].plot(V, bh, color=COLORS['accent'], linewidth=2.5, linestyle='--')
axes[1, 2].set_title(r'$\beta_h(V)$ — Na$^+$ inactivation closing')
axes[1, 2].set_xlabel('Membrane potential $V$ (mV)')

# Add resting potential line to all subplots
for ax in axes.flat:
    ax.axvline(-65, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)

fig.suptitle('Hodgkin-Huxley Rate Functions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Plot steady-state gating variables m_inf, h_inf, n_inf vs voltage

V = np.linspace(-100, 50, 1000)

m_ss = np.array([m_inf(v) for v in V])
h_ss = np.array([h_inf(v) for v in V])
n_ss = np.array([n_inf(v) for v in V])

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(V, m_ss, color=COLORS['Na'], linewidth=2.5, label=r'$m_\infty(V)$ — Na$^+$ activation')
ax.plot(V, h_ss, color=COLORS['accent'], linewidth=2.5, label=r'$h_\infty(V)$ — Na$^+$ inactivation')
ax.plot(V, n_ss, color=COLORS['K'], linewidth=2.5, label=r'$n_\infty(V)$ — K$^+$ activation')

# Product m_inf^3 * h_inf shows the steady-state Na+ conductance fraction
ax.plot(V, m_ss**3 * h_ss, color=COLORS['Na'], linewidth=1.5, linestyle='--',
        alpha=0.7, label=r'$m_\infty^3 \cdot h_\infty$ — steady-state Na$^+$ window')

ax.axvline(-65, color='gray', linewidth=0.8, linestyle=':', alpha=0.7, label='$V_{\\mathrm{rest}}$')
ax.set_xlabel('Membrane potential $V$ (mV)')
ax.set_ylabel('Steady-state value')
ax.set_title('Steady-State Gating Variables')
ax.legend(loc='center right', framealpha=0.9)
ax.set_ylim(-0.02, 1.05)
ax.set_xlim(-100, 50)

plt.tight_layout()
plt.show()

In [ ]:
# Plot time constants tau_m, tau_h, tau_n vs voltage

V = np.linspace(-100, 50, 1000)

tm = np.array([tau_m(v) for v in V])
th = np.array([tau_h(v) for v in V])
tn = np.array([tau_n(v) for v in V])

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(V, tm, color=COLORS['Na'], linewidth=2.5,
        label=r'$\tau_m(V)$ — Na$^+$ activation')
ax.plot(V, th, color=COLORS['accent'], linewidth=2.5,
        label=r'$\tau_h(V)$ — Na$^+$ inactivation')
ax.plot(V, tn, color=COLORS['K'], linewidth=2.5,
        label=r'$\tau_n(V)$ — K$^+$ activation')

ax.axvline(-65, color='gray', linewidth=0.8, linestyle=':', alpha=0.7, label='$V_{\\mathrm{rest}}$')
ax.set_xlabel('Membrane potential $V$ (mV)')
ax.set_ylabel('Time constant $\\tau$ (ms)')
ax.set_title('Voltage-Dependent Time Constants of Gating Variables')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_xlim(-100, 50)

# Annotate the key observation
ax.annotate(r'$\tau_m \ll \tau_n, \tau_h$: Na$^+$ activation is much faster',
            xy=(-40, 0.5), fontsize=11, fontstyle='italic', color=COLORS['Na'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['Na'], alpha=0.8))

plt.tight_layout()
plt.show()

## 3.5 The Complete Model

We can now assemble the full Hodgkin-Huxley model. The state of the membrane patch at any instant is described by four variables: the membrane potential $V$ and the three gating variables $n$, $m$, $h$. Their dynamics are governed by the following system of **four coupled ordinary differential equations**:

$$\boxed{\begin{aligned}
C_m \frac{dV}{dt} &= -\bar{g}_\text{Na} \, m^3 h \, (V - E_\text{Na}) - \bar{g}_\text{K} \, n^4 \, (V - E_\text{K}) - g_L \, (V - E_L) + I_\text{ext} \\[6pt]
\frac{dn}{dt} &= \alpha_n(V)(1 - n) - \beta_n(V) \, n \\[4pt]
\frac{dm}{dt} &= \alpha_m(V)(1 - m) - \beta_m(V) \, m \\[4pt]
\frac{dh}{dt} &= \alpha_h(V)(1 - h) - \beta_h(V) \, h
\end{aligned}}$$

The coupling is bidirectional and nonlinear: the gating variables affect $V$ through the conductances $m^3 h$ and $n^4$, while $V$ affects the gating variables through the rate functions $\alpha_x(V)$ and $\beta_x(V)$. This feedback is what produces the rich dynamics of the model — including the action potential.

### Parameter Table

The following are the **original parameters** from Hodgkin and Huxley (1952), measured at $6.3\,°\text{C}$:

| Parameter | Symbol | Value | Units |
|-----------|--------|-------|-------|
| Membrane capacitance | $C_m$ | 1.0 | $\mu\text{F/cm}^2$ |
| Max Na$^+$ conductance | $\bar{g}_\text{Na}$ | 120.0 | $\text{mS/cm}^2$ |
| Max K$^+$ conductance | $\bar{g}_\text{K}$ | 36.0 | $\text{mS/cm}^2$ |
| Leak conductance | $g_L$ | 0.3 | $\text{mS/cm}^2$ |
| Na$^+$ reversal potential | $E_\text{Na}$ | +50.0 | mV |
| K$^+$ reversal potential | $E_\text{K}$ | $-77.0$ | mV |
| Leak reversal potential | $E_L$ | $-54.4$ | mV |

Note the striking asymmetry: $\bar{g}_\text{Na}$ is more than three times larger than $\bar{g}_\text{K}$, which in turn is 120 times larger than $g_L$. When the Na$^+$ channels open fully, they can drive an enormous inward current that rapidly depolarizes the membrane.

These seven parameters, together with the six rate functions defined in Section 3.4, completely specify the model. In the next sections, we will simulate this system numerically and explore its remarkably rich dynamical behavior.